In [1]:
# 1. Install required packages silently in Kaggle
!pip install -q --upgrade bitsandbytes trl peft transformers datasets

# 2. Download the utility file containing the evaluate function
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import PeftModel
from datasets import load_dataset
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from util import evaluate

# 3. Mode Configuration
LITE_MODE = False  
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct" 
HUB_MODEL_NAME = "sairuthwik112/price-prediction-qwen-final"
DATASET_NAME = "ed-donner/items_prompts_lite" if LITE_MODE else "ed-donner/items_prompts_full"

# 4. Fetch the token securely using Kaggle Secrets
print("Retrieving Hugging Face token from Kaggle Secrets...")
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN secret not found! Please add it to your Kaggle Add-ons -> Secrets menu.")

login(token=hf_token, add_to_git_credential=True)

# 5. Load the Test Split of the Lite Dataset
print(f"\nLoading dataset: {DATASET_NAME}...")
dataset = load_dataset(DATASET_NAME)
test = dataset['test']

# 6. Setup Quantization (Crucial for running smoothly on Kaggle T4 GPUs)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 7. Load Tokenizer and Base Model
print(f"Loading base model: {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto" 
)

# 8. Load Fine-Tuned PEFT Adapter weights from your HF repository
print(f"Loading fine-tuned adapter: {HUB_MODEL_NAME}...")
fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)
print(f"Model memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

# 9. Prediction Function matching your setup
def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    
    # Extract only the newly generated prediction tokens (ignore input prompt tokens)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

# 10. Execute the Evaluation Loop
print("\n--- Starting Evaluation in Kaggle ---")
set_seed(42)
evaluate(model_predict, test)